# MIDUS 3 (ICPSR 38837): ¿el BSI en reposo predice la respuesta al reto de laboratorio?
**EXPLORATORIO.** No prueba el techo adaptativo (PE3): aquí hay un solo reto por persona.
**BSI aquí = 3 dominios** (inflamación, metabólico, neuroendocrino con cortisol urinario y DHEA-S). **No hay reloj epigenético en MIDUS.**

**Reglas fijadas ANTES de ver resultados**
- Resultado PRIMARIO: reactividad de HRV (rMSSD log, matemáticas menos línea base). Los otros 3 son exploratorios.
- Todos los modelos incluyen edad, sexo, IMC, nivel basal del resultado, antihipertensivos y betabloqueantes.
- M2 suma carga alostatica; M3 suma el BSI promedio; M4 suma los 11 marcadores continuos (control); M5 suma los 3 dominios por separado.
- Validación cruzada 5x5 y bootstrap pareado de 2000 réplicas sobre ΔR² (fuera de muestra).
- **SUMA**: límite inferior del IC95% de ΔR² > 0. **NO SUMA**: IC95% completamente dentro de ±0.01. Otro caso: **INCONCLUSO**.
- Solo casos completos; se imprime el n de cada análisis.

In [1]:
# 1) Sube el zip de datos de ICPSR (el de 96 MB, formato Delimitado). Espera a que termine la barra.
from google.colab import files
up = files.upload()
print(list(up.keys()))

Saving ICPSR_38837-V1.zip to ICPSR_38837-V1.zip
['ICPSR_38837-V1.zip']


In [2]:
# 2) Descomprime y busca el archivo de datos
import zipfile, glob, os
for z in glob.glob('*.zip'):
    zipfile.ZipFile(z).extractall('midus_data')
cands = [f for f in glob.glob('midus_data/**/*', recursive=True) if f.lower().endswith(('.tsv','.txt','.csv')) and 'DS0001' in f and 'Data' in os.path.basename(f)]
print(cands)
assert len(cands) >= 1, 'No encontre el archivo de datos. Mandame lo que imprime la linea de arriba y esta lista:'
DATA = cands[0]; print('Usare:', DATA)

['midus_data/ICPSR_38837/DS0001/38837-0001-Data.tsv']
Usare: midus_data/ICPSR_38837/DS0001/38837-0001-Data.tsv


In [3]:
%%writefile midus_bsi.py
"""MIDUS 3 (ICPSR 38837): ¿el BSI en reposo predice la respuesta al reto de laboratorio
mas alla de edad, sexo, IMC, nivel basal, medicamentos y carga alostatica?
REGLAS FIJADAS ANTES DE VER RESULTADOS (exploratorio; no prueba el techo adaptativo PE3)."""
import numpy as np, pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score

SEED, N_BOOT, MARGIN = 2026, 2000, 0.01
RULE = ("SUMA: limite inferior del IC95% de dR2 > 0. NO SUMA: IC95% de dR2 dentro de (-0.01,+0.01). "
        "Otro caso: INCONCLUSO. Resultado PRIMARIO: reactividad de HRV (rMSSD log) en la prueba de matematicas. "
        "Los demas resultados son EXPLORATORIOS (sin correccion por comparaciones multiples).")

MISS = {  # codigos de dato faltante segun el libro de codigos ICPSR
 "C4BCRP":(98,99),"C4BIL6":(98,99),"C4BFGN":(998,999),"C4BSICAM":(9998,9999),"C4BGLUC":(998,999),
 "C4BINSLN":(998,999),"C4BTRIGL":(9998,9999),"C4BHDL":(998,999),"C4BHA1C":(98,99),"C4BDHEAS":(998,999),
 "C4BCLCRE":(998,999),"C4BSCL2":(998,999),"C4BSCL3":(998,999),"C4PBMI":(97,98,99),
 "C4VB1LRM":(8,9),"C4VB2LRM":(8,9),"C4VM1LRM":(8,9),"C4VU1LRM":(8,9),
 "C4VSB1BP":(9998,9999),"C4VSB2BP":(9998,9999),"C4VSM1BP":(9998,9999),
 "C4XPC_1156":(8,9),"C4XPC_1291":(8,9)}
NEEDED = ["M2ID","C1PRSEX","C4ZAGE"] + list(MISS)

def load(path):
    df = pd.read_csv(path, sep="\t", low_memory=False)
    df.columns = [c.strip().upper() for c in df.columns]
    miss = [c for c in NEEDED if c not in df.columns]
    if miss: raise SystemExit(f"FALTAN COLUMNAS: {miss}. Detente y mandame esta lista.")
    for c, codes in MISS.items():
        df[c] = pd.to_numeric(df[c], errors="coerce"); df.loc[df[c].isin(codes), c] = np.nan
    for c in ["C4ZAGE","C1PRSEX"]: df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

zs = lambda x: (x - x.mean()) / x.std(ddof=0)

def build(df):
    d = pd.DataFrame({"id": df["M2ID"]})
    d["age"] = df["C4ZAGE"]; d["female"] = (df["C1PRSEX"] == 2).astype(int); d["bmi"] = df["C4PBMI"]
    d["antihyp"] = (df["C4XPC_1156"] == 1).astype(float).where(df["C4XPC_1156"].notna())
    d["betablk"] = (df["C4XPC_1291"] == 1).astype(float).where(df["C4XPC_1291"].notna())
    L = lambda c, lo=1e-3: np.log(df[c].clip(lower=lo))
    m = pd.DataFrame({"crp":L("C4BCRP"),"il6":L("C4BIL6"),"fgn":L("C4BFGN"),"sicam":L("C4BSICAM"),
        "glu":df["C4BGLUC"],"ins":L("C4BINSLN",0.1),"tg":L("C4BTRIGL",1),"hba1c":df["C4BHA1C"],"hdl":df["C4BHDL"],
        "ucort":L("C4BCLCRE"),"dheas":L("C4BDHEAS")})
    d = pd.concat([d, m.add_prefix("m_")], axis=1)
    d["dom_inflam"] = zs(pd.concat([zs(m.crp),zs(m.il6),zs(m.fgn),zs(m.sicam)],axis=1).mean(axis=1))
    d["dom_metab"]  = zs(pd.concat([zs(m.glu),zs(m.ins),zs(m.tg),zs(m.hba1c),-zs(m.hdl)],axis=1).mean(axis=1))
    d["dom_neuro"]  = zs(pd.concat([zs(m.ucort),-zs(m.dheas)],axis=1).mean(axis=1))
    d["bsi_mean"] = d[["dom_inflam","dom_metab","dom_neuro"]].mean(axis=1)
    top = [m.crp,m.il6,m.fgn,m.hba1c,m.tg,m.glu,m.ins]
    risk = pd.concat([x > x.quantile(.75) for x in top] + [m.hdl < m.hdl.quantile(.25)], axis=1)
    d["allo_load"] = risk.sum(axis=1)
    # resultados (respuesta al reto) y su nivel basal
    hb = df[["C4VB1LRM","C4VB2LRM"]].mean(axis=1); sb = df[["C4VSB1BP","C4VSB2BP"]].mean(axis=1)
    d["hrv_base"], d["hrv_react"] = hb, df["C4VM1LRM"] - hb
    d["sbp_base"], d["sbp_react"] = sb, df["C4VSM1BP"] - sb
    d["cort_base"] = np.log(df["C4BSCL2"].clip(lower=1e-3)); d["cort_react"] = np.log(df["C4BSCL3"].clip(lower=1e-3)) - d["cort_base"]
    d["hrv_orth"] = df["C4VU1LRM"] - hb
    return d

OUTCOMES = {  # nombre: (columna resultado, columna basal, primario?)
 "PRIMARIO hrv_react (HRV, matematicas)": ("hrv_react","hrv_base"),
 "exploratorio sbp_react (PAS, matematicas)": ("sbp_react","sbp_base"),
 "exploratorio cort_react (cortisol, post-reto)": ("cort_react","cort_base"),
 "exploratorio hrv_orth (HRV, de pie)": ("hrv_orth","hrv_base")}
MK = ["m_crp","m_il6","m_fgn","m_sicam","m_glu","m_ins","m_tg","m_hba1c","m_hdl","m_ucort","m_dheas"]

def models(base):
    c = ["age","female","bmi",base,"antihyp","betablk"]
    return {"M1 base": c, "M2 +carga alostatica": c+["allo_load"], "M3 +BSI promedio": c+["allo_load","bsi_mean"],
            "M4 +11 marcadores continuos": c+["allo_load"]+MK,
            "M5 +3 dominios separados": c+["allo_load","dom_inflam","dom_metab","dom_neuro"]}

def oof(X, y, folds=5, reps=5):
    out = np.zeros((reps, len(y)))
    for r in range(reps):
        for tr, te in KFold(folds, shuffle=True, random_state=SEED+r).split(X):
            mu, sd = X[tr].mean(0), X[tr].std(0); sd[sd==0] = 1
            out[r, te] = LinearRegression().fit((X[tr]-mu)/sd, y[tr]).predict((X[te]-mu)/sd)
    return out.mean(0)

def dboot(y, pn, po):
    rng = np.random.default_rng(SEED); n = len(y); ds = []
    for _ in range(N_BOOT):
        i = rng.integers(0, n, n); ds.append(r2_score(y[i], pn[i]) - r2_score(y[i], po[i]))
    return np.percentile(ds, [2.5, 97.5])

def verdict(lo, hi): return "SUMA" if lo > 0 else ("NO SUMA" if (lo > -MARGIN and hi < MARGIN) else "INCONCLUSO")

def run_outcome(d, ycol, bcol):
    M = models(bcol); cols = sorted({c for v in M.values() for c in v} | {ycol})
    z = d[cols].dropna().reset_index(drop=True); y = z[ycol].values
    P = {k: oof(z[v].values, y) for k, v in M.items()}
    tab = pd.DataFrame({"modelo": list(P), "R2 fuera de muestra": [round(r2_score(y, p), 4) for p in P.values()]})
    res = {"n": len(z), "tabla": tab}
    for name, a, b in [("M3 vs M2 (BSI suma a carga alostatica?)","M3 +BSI promedio","M2 +carga alostatica"),
                       ("M4 vs M3 (marcadores continuos vs BSI)","M4 +11 marcadores continuos","M3 +BSI promedio"),
                       ("M5 vs M2 (3 dominios separados)","M5 +3 dominios separados","M2 +carga alostatica")]:
        lo, hi = dboot(y, P[a], P[b]); dr = r2_score(y, P[a]) - r2_score(y, P[b])
        res[name] = {"dR2": round(dr, 4), "IC95%": (round(lo, 4), round(hi, 4)), "veredicto": verdict(lo, hi)}
    return res


Writing midus_bsi.py


In [4]:
# 3) Analisis (tarda ~1-3 min)
import importlib, midus_bsi as B, pandas as pd
importlib.reload(B)
d = B.build(B.load(DATA))
print(B.RULE, '\n')
resumen = []
for nombre, (y, b) in B.OUTCOMES.items():
    r = B.run_outcome(d, y, b)
    print('='*70); print(nombre, '| n =', r['n']); print(r['tabla'].to_string(index=False))
    for k, v in r.items():
        if k.startswith('M'):
            print(' ', k, '->', v); resumen.append({'resultado': nombre.split()[1], 'n': r['n'], 'comparacion': k[:8], **v})
print('\nRESUMEN'); print(pd.DataFrame(resumen).to_string(index=False))

SUMA: limite inferior del IC95% de dR2 > 0. NO SUMA: IC95% de dR2 dentro de (-0.01,+0.01). Otro caso: INCONCLUSO. Resultado PRIMARIO: reactividad de HRV (rMSSD log) en la prueba de matematicas. Los demas resultados son EXPLORATORIOS (sin correccion por comparaciones multiples). 

PRIMARIO hrv_react (HRV, matematicas) | n = 521
                     modelo  R2 fuera de muestra
                    M1 base               0.1023
       M2 +carga alostatica               0.1098
           M3 +BSI promedio               0.1102
M4 +11 marcadores continuos               0.0862
   M5 +3 dominios separados               0.1049
  M3 vs M2 (BSI suma a carga alostatica?) -> {'dR2': 0.0004, 'IC95%': (np.float64(-0.0082), np.float64(0.009)), 'veredicto': 'NO SUMA'}
  M4 vs M3 (marcadores continuos vs BSI) -> {'dR2': -0.0239, 'IC95%': (np.float64(-0.0437), np.float64(-0.0057)), 'veredicto': 'INCONCLUSO'}
  M5 vs M2 (3 dominios separados) -> {'dR2': -0.0049, 'IC95%': (np.float64(-0.0185), np.float64(0.00